In [ ]:
import pandas as pd
import numpy as np
import ast

from sklearn.model_selection import GroupShuffleSplit

PATH_CLIPS                          = "../Data/Videos/Clips/"
PATH_KEYPOINTS                      = "../Data/Unprocessed/keypoints.csv"
PATH_METRICS                        = "../Data/Unprocessed/metrics.csv"
PATH_ACTIONS_FILTERED               = "../Data/Processed/actions_filtered.csv"

PATH_METADATA                        = "../Data/Processed/metadata.csv"
PATH_TRAIN                           = "../Data/Processed/train.csv"
PATH_TEST                            = "../Data/Processed/test.csv"

WINDOW_SIZE                         = 4
WINDOWS_PER_ACTION                  = 5
NUM_JOINTS                          = 17

In [189]:
df_metrics = pd.read_csv(PATH_METRICS)

total_expected_frames = df_metrics["expected"].sum()
total_actual_frames = df_metrics["actual"].sum()
total_coverage = total_actual_frames / total_expected_frames

print(df_metrics[df_metrics["expected"] != df_metrics["actual"]])
print("")

print("Expected frames: ", total_expected_frames)
print("Actual frames:   ", total_actual_frames)
print(f"Coverage:         {total_coverage*100:.2f}%")

    fencer  action_id  start_frame  end_frame  expected  actual   coverage
29    LEFT        170           22         29         8       7  87.500000
41    LEFT        209           22         29         8       7  87.500000
44   RIGHT        219           24         30         7       5  71.428571
65   RIGHT        142           30         37         8       6  75.000000
67   RIGHT        208           29         36         8       6  75.000000
..     ...        ...          ...        ...       ...     ...        ...
430  RIGHT        298           27         44        18      17  94.444444
439  RIGHT        284           16         22         7       6  85.714286
441  RIGHT        280           16         21         6       5  83.333333
446   LEFT        316           28         35         8       7  87.500000
447  RIGHT        317           24         34        11       9  81.818182

[67 rows x 7 columns]

Expected frames:  4247
Actual frames:    4138
Coverage:         97.43%


In [190]:
df_keypoints = pd.read_csv(PATH_KEYPOINTS)

df_keypoints["box"] = df_keypoints["box"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

df_keypoints["keypoints"] = df_keypoints["keypoints"].apply(
    lambda x: [tuple(p) for p in ast.literal_eval(x)] if isinstance(x, str) else x
)

In [191]:
df_filtered = pd.read_csv(PATH_ACTIONS_FILTERED)

df_merged = df_keypoints.merge(df_filtered, on=["file", "fencer"], how="left")
df_merged = df_merged[
    (df_merged["frame"] >= df_merged["start_frame"]) &
    (df_merged["frame"] <= df_merged["end_frame"])
]

df_merged = df_merged[["file", "fencer", "action_id", "action", "frame", "start_frame", "end_frame", "box", "confidence", "keypoints"]].reset_index(drop=True)
print(df_merged)

               file fencer  action_id        action  frame  start_frame  \
0     1/10_Left.mp4   LEFT          0  ATTACK_LUNGE     23           23   
1     1/10_Left.mp4   LEFT          0  ATTACK_LUNGE     24           23   
2     1/10_Left.mp4   LEFT          0  ATTACK_LUNGE     25           23   
3     1/10_Left.mp4   LEFT          0  ATTACK_LUNGE     26           23   
4     1/10_Left.mp4  RIGHT          1  ATTACK_LUNGE     21           21   
...             ...    ...        ...           ...    ...          ...   
4202   6/9_Left.mp4  RIGHT        468  ATTACK_BASIC     40           38   
4203   6/9_Left.mp4  RIGHT        468  ATTACK_BASIC     41           38   
4204   6/9_Left.mp4  RIGHT        468  ATTACK_BASIC     42           38   
4205   6/9_Left.mp4  RIGHT        468  ATTACK_BASIC     43           38   
4206   6/9_Left.mp4  RIGHT        468  ATTACK_BASIC     44           38   

      end_frame                     box  confidence  \
0            26    (749, 585, 885, 791)    0

In [192]:
gss = GroupShuffleSplit(
    n_splits=1,
    train_size=0.8,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(df_merged, groups=df_merged['action_id'])
)

df_train = df_merged.iloc[train_idx]
df_test  = df_merged.iloc[test_idx]

In [193]:
action_counts = df_train.groupby("action")["action_id"].nunique().sort_values(ascending=False)
max_count = action_counts.max()

class_weights = max_count / action_counts
class_weights["ATTACK_LUNGE"] = 4
class_weights["ATTACK_COUNTER"] = 6.6
class_weights["DEFENSE_DISTANCE_PULL"] = 6.7
weights_dict = class_weights.to_dict()

print(action_counts)
print("")
print(class_weights)

action
ATTACK_LUNGE             169
ATTACK_BASIC             100
DEFENSE_DISTANCE_PULL     54
DEFENSE_POINT_IN_LINE     23
DEFENSE_PARRY             17
ATTACK_FLUNGE              9
Name: action_id, dtype: int64

action
ATTACK_LUNGE              4.000000
ATTACK_BASIC              1.690000
DEFENSE_DISTANCE_PULL     6.700000
DEFENSE_POINT_IN_LINE     7.347826
DEFENSE_PARRY             9.941176
ATTACK_FLUNGE            18.777778
ATTACK_COUNTER            6.600000
Name: action_id, dtype: float64


In [194]:
def create_action_windows(df, window_size=4, num_windows=4, class_weights=None, random_state=42):
    rng = np.random.default_rng(random_state)
    window_rows = []
    window_counter = 0

    # Sort for safety
    df = df.sort_values(["action_id", "frame"]).reset_index(drop=True)

    for _, group in df.groupby(["action_id"]):
        action = group["action"].iloc[0]

        # Determine number of windows for this action
        weight = class_weights.get(action, 1.0) if class_weights else 1.0
        windows_per_action = max(1, int(round(num_windows * weight)))

        frames = group["frame"].values
        max_start = len(frames) - window_size
        if max_start < 0:
            continue  # action too short for a single window

        start_indices = rng.choice(
            np.arange(0, max_start + 1),
            size=windows_per_action,
            replace=False if windows_per_action <= max_start + 1 else True
        )

        for start in start_indices:
            window_counter += 1
            window_id = window_counter

            window_slice = group.iloc[start:start + window_size].copy()
            window_slice["window_id"] = window_id
            window_rows.append(window_slice)

    return pd.concat(window_rows, ignore_index=True)

In [195]:
df_train_windowed = create_action_windows(df_train, window_size=WINDOW_SIZE, num_windows=WINDOWS_PER_ACTION, class_weights=weights_dict) # Class weights unnecessary for now
df_test_windowed  = create_action_windows(df_test, window_size=WINDOW_SIZE, num_windows=WINDOWS_PER_ACTION)

train_counts = df_train_windowed.groupby("action")["window_id"].nunique().sort_values(ascending=False)
test_counts  = df_test_windowed.groupby("action")["window_id"].nunique().sort_values(ascending=False)

print(train_counts)
print("")
print(test_counts)

print("")
print("Number of training action snippets: ", df_train_windowed["window_id"].nunique())
print("Number of testing action snippets:  ", df_test_windowed["window_id"].nunique())

action
ATTACK_LUNGE             3380
DEFENSE_DISTANCE_PULL    1836
DEFENSE_POINT_IN_LINE     851
DEFENSE_PARRY             850
ATTACK_FLUNGE             846
ATTACK_BASIC              800
Name: window_id, dtype: int64

action
ATTACK_LUNGE             210
ATTACK_BASIC             135
DEFENSE_DISTANCE_PULL     55
DEFENSE_POINT_IN_LINE     40
DEFENSE_PARRY             25
ATTACK_FLUNGE              5
Name: window_id, dtype: int64

Number of training action snippets:  8563
Number of testing action snippets:   470


In [200]:
def explode_keypoints(df):
    # Expand each tuple into separate x,y columns
    exploded = df["keypoints"].apply(
        lambda kp: [coord for point in kp for coord in point]
    )

    # Create column names: x0, y0, x1, y1, ...
    num_points = len(df.iloc[0]["keypoints"])
    cols = [f"x{i}" for i in range(num_points)] + [f"y{i}" for i in range(num_points)]
    
    new_df = pd.DataFrame(exploded.tolist(), columns=cols)
    return pd.concat([df.drop(columns=["keypoints"]), new_df], axis=1)

In [ ]:
df_metadata_train = df_train_windowed[["file", "fencer", "action_id", "window_id", "action", "frame", "start_frame", "end_frame", "box"]]
df_metadata_test = df_test_windowed[["file", "fencer", "action_id", "window_id", "action", "frame", "start_frame", "end_frame", "box"]]

df_metadata = pd.concat([df_metadata_train, df_metadata_test], ignore_index=True)
df_metadata.sort_values(["action_id", "window_id", "frame"], inplace=True)

df_train_data = df_train_windowed[["window_id", "frame", "action", "confidence", "keypoints"]].copy()
df_test_data = df_test_windowed[["window_id", "frame", "action", "confidence", "keypoints"]].copy()

df_train_data.sort_values(["window_id"], inplace=True)
df_test_data.sort_values(["window_id"], inplace=True)

df_train_data = explode_keypoints(df_train_data)
df_test_data = explode_keypoints(df_test_data)

df_train_data.to_csv(PATH_TRAIN, index=False)
df_test_data.to_csv(PATH_TEST, index=False)
df_metadata.to_csv(PATH_METADATA, index=False)